<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Encoders: Named Entity Recognition(NER) - Solution

---

### About
Often you want to identify companies, people, or places to track stocks, or sort customer service tickets. In this notebook, you will apply encoder models to explore the functionality of **Named Entity Recognition (NER)** to identify entities in text.

### Learning Objective
Apply Enocoder architecture to a Named Entity Recognition (NER) task.

### Notebook Guide
- NLP Scenario
- Initialize the pipeline for your task: Named Entity Recognition (NER)
- Let's modify our pipeline to use whole words
- Try It!
- Conclusions and Takeaways

### Imports

In [ ]:
%pip install -qqq transformers


In [ ]:
#imports 
import numpy as np
import pandas as pd

from transformers import pipeline

# NLP Scenario 
You are an analyst for a marketing company that just launched a new product suite of mobile devices. You have data from product reviews of the new product and need to identify the entities in the reviews. 

#####  Product Reviews 
1. "I absolutely love the TechWave X1! It has made my daily tasks in Spain so much easier and more efficient. Highly recommend it!"
2. "I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow here in Mexico."
3. "The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine."
4. "I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it."
5. "The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it."
6.  "The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities."

In [ ]:
# data setup 
# create a list of reviews 
reviews = ["I absolutely love the TechWave X1! It has made my daily tasks so much easier and more efficient. Highly recommend it!",
"I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow.",
"The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine.",
"I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it.",
"The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it.",
"The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities."
]

# Initialize the pipeline for your task: Named Entity Recognition (NER)

The default analyzer for NER analysis can be called with: 
```python 
    pipeline("ner")
```
This pipeline can take data at sentence, paragraph, or document level and will return:
- entity- classification (see below for entity classification types)
- score- confidence of classification (range 0-1) where 1 is highly confident 
- index- token position 
- word - actual text, may be a subword (##subword)
- start/end- character position 

All the models we use here have the encoder architecture. You could also consider an Encoder-Decoder model, but it is likely not needed for this task. 

#### Entity Types: 
- `O` means the word doesn’t correspond to any entity.
- `B-PER/I-PER` means the word corresponds to the beginning of/is inside a person entity.
- `B-ORG/I-ORG` means the word corresponds to the beginning of/is inside an organization entity.
- `B-LOC/I-LOC` means the word corresponds to the beginning of/is inside a location entity.
- `B-MISC/I-MISC` means the word corresponds to the beginning of/is inside a miscellaneous entity.  
source: [Hugging Face](https://huggingface.co/learn/nlp-course/en/chapter7/2?fw=pt)

In [ ]:
# initalize the NER pipeline
ner_pipeline = pipeline("ner")

While you're waiting for the model to download, go and look at it's model card in Hugging Face.

#### BBC example 

On 09 Dec 24 the BBC reported 
```python
"""
Shares in US chocolate maker Hershey have jumped by more than 10% after 
a report that Mondelez International, which owns UK-based Cadbury, 
has approached the firm about a potential buyout.
"""
```
[source](https://www.bbc.com/news/articles/c62w62vydnvo)

#### Let's test our NER pipeline on this sample text 

In [ ]:
# load sample news text 
business_news = """
Shares in US chocolate maker Hershey have jumped by more than 10% after 
a report that Mondelez International, which owns UK-based Cadbury, 
has approached the firm about a potential buyout.
"""

In [ ]:
# run pipeline and review the results   
results = ner_pipeline(business_news)
results

In [ ]:
# print the entities
for entity in results:
    print(f"Entity: {entity['entity']}, Value: {entity['word']}")

#### Results 
- Notice that Hershey, Mondelez International, and Cadbury are all correctly identified as organizations. 
- Each of them are split into subwords. For example, Hershey is split into three tokens: "her", "##she",and "##y".
  - This is because the model uses a technique called WordPiece tokenization, which splits words into subwords when necessary. This allows the model to handle out-of-vocabulary words and improve generalization.
- In contrast US is correctly identified as a location, and UK is identified as a miscellaneous entity. 

# Let's modify our pipeline to use whole words
There are several ways to do this, but one of the easiest is to change the aggregation strategy to simple

```python 
ner_pipeline = pipeline("ner", aggregation_strategy="simple") 
```

aggregation_strategy="simple"

In [ ]:
# initalize the NER pipeline with aggregation_strategy set to simple
ner_simple = pipeline("ner", aggregation_strategy="simple")

In [ ]:
ner_simple(business_news)

#### Simple Aggregation Strategy results
Notice that now we have a single entity for each word in the text. This is because the simple aggregation strategy returns each word as a separate entity. Notice that we get "LOC", "ORG", "MISC" instead of B/I-LOC

# Try It!
- Run the `ner_pipeline` pipeline on our `reviews.csv` and see what entities are detected in the reviews.
- Run the `ner_simple` pipeline on the reviews list and see what entities are detected in the reviews.
- Compare the results of the two pipelines. What differences do you notice? 
- How well did our model do at detecting new entities in the reviews?

In [ ]:
# run the ner_pipeline we created on the reviews list and see what entities are detected in the reviews.
ner_pipeline(reviews)

In [ ]:
# run the ner_simple pipeline on the reviews list and see what entities are detected in the reviews.
ner_simple(reviews)

- Compare the results of the two pipelines. What differences do you notice? 
- How well did our model do at detecting new entities in the reviews?


A: Both models did well at detecting our new product as entity and labeling it as MISC. The simple model returned one entity group per review instead of subwords. 

#### BONUS: Try switching to a named model from the Hugging Face hub! 

In [ ]:
# new_ner = pipeline("ner", model="jean-baptiste/roberta-large-ner-english", aggregation_strategy="simple")
# or new_ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

In [ ]:
# new_ner(reviews)

# Conclusions and Takeaways
- Named Entity Recognition can be a powerful tool for finding/tracking people, places, and things (such as companies) in text
- This has many useful applications in business and finance (tracking company news, regulatory changes, etc.), as well as biotech (finding drugs, genes, etc.)
- This model is powerful when it is paired with other models (for example tracking sentiment by entity)